# Lab 5: Sampling, Intervals, and Error

<img src="ds4all-logo.png" alt="Data Science for All Logo" style="width: 15%; float: right; padding: 1%; margin-right: 2%;"/>

Welcome to Lab 5 of *Data Science for All*!

In lecture you built confidence intervals from a single sample and tested claims with p-values. In
this lab we check how well those methods work. Our data this week is a whole population, so we know
the true answer and can see how often a sample gets close to it, and how often it doesn't.

This lab will be due on **Thursday at 5 PM (Week 11)**. As always, collaborating on labs is encouraged,
but don't just share answers.

## Today's lab

In today's lab, you'll learn how to:

1. simulate thousands of samples, and see why their means pile up in a bell shape even when the data don't;
2. measure how the spread of the sample means shrinks as the sample grows;
3. estimate a standard error and build a 95% confidence interval from one sample, with the bootstrap and with a formula, as in Coding Assignment 4; and
4. count how often a 95% interval misses, and see what that means when you use one to test a claim.

## Resources

Keep these open while you work:

- **Class notes:** [Sampling and Uncertainty](https://dats.columbian.gwu.edu/ds4all/modules/module-3.4.html), [Single Proportion Tests](https://dats.columbian.gwu.edu/ds4all/modules/module-3.5.html), and the [single proportion test notebook](https://gwu.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fsarahcburnett%2Fds4all-materials-f26-public&branch=main&urlpath=tree%2Fds4all-materials-f26-public%2Flecture-notebooks%2Fstats%2Fhypothesis_testing_single_prop.ipynb)
- **infer documentation:** [infer.tidymodels.org](https://infer.tidymodels.org/)
- **Reading:** [IMS, Chapter 12: Confidence intervals with bootstrapping](https://openintro-ims.netlify.app/foundations-bootstrapping), and from the Data 8 textbook, [14.5: The Variability of the Sample Mean](https://dats.columbian.gwu.edu/ds4all/modules/mean.html#the-variability-of-the-sample-mean) and [11.4: Error Probabilities](https://dats.columbian.gwu.edu/ds4all/modules/testing-hypotheses.html#error-probabilities)

These are the tools you'll use most:

| If you want to... | Use |
|---|---|
| draw one random sample of rows from a tibble | `slice_sample(n = ..., replace = TRUE)` |
| draw many random samples at once | `rep_slice_sample(n = ..., replace = TRUE, reps = ...)` |
| compute one value per sample | `summarize()`, right after `rep_slice_sample()` |
| bootstrap a sample | `specify()`, then `generate(type = "bootstrap")`, then `calculate()` |
| find a percentile | `quantile(x, 0.025)` |
| measure spread | `sd()` |

In [ ]:
# Initialize the autograder and JupyterHub helper
source("helpers.R")

library(readr)
library(dplyr)
library(ggplot2)
library(infer)

options(repr.matrix.max.rows = 20)
options(repr.plot.width = 7, repr.plot.height = 4.5)

As in Lab 4, we load only the packages that Coding Assignment 4's autograder has, so anything that
runs here will also run there.

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


## 1. A population, and the means of samples from it

Our data this week: the 2023 salaries of 41,252 employees of the City and County of San Francisco, one
row per employee. Each salary adds up the employee's regular pay, overtime, and other pay. The file
comes from Data 8, UC Berkeley's introductory data science course, which got it from
[Transparent California](https://transparentcalifornia.com/salaries/2023/san-francisco/).

<div style="border-left: 4px solid #fdb515; background: #fffdf5; padding: 0.8em 1em; margin: 1em 0;">
<b>In this lab, the data is the population.</b>
<p>In practice you almost never have data on everyone. You estimate from a sample, and you never learn
the true answer. Here we have every employee, so we can draw samples, make estimates, and then check
them against the truth.</p>
</div>

In [ ]:
salaries <- read_csv("data/sf_salaries_2023.csv", show_col_types = FALSE)

salaries

A number that describes the whole population, like the average salary of all 41,252 employees, is a
**parameter**. A number computed from a sample is a **statistic**. Usually the parameter is unknown,
and a statistic is our estimate of it.

---

**Question 1.1.** Compute the population's average salary and its standard deviation (SD). Store each
as a single number, in `pop_mean` and `pop_sd`.

In [ ]:
pop_mean <- NULL # YOUR CODE HERE
pop_sd   <- NULL # YOUR CODE HERE

c(pop_mean, pop_sd)

Now make a histogram of all 41,252 salaries, named `salary_hist`. Fill in the blanks. The red line
marks `pop_mean`, and `scale_x_continuous()` writes the axis in dollars; both are done for you.

In [ ]:
salary_hist <- ggplot(salaries) +
  aes(x = ...) +
  geom_...(binwidth = 10000, fill = "chartreuse4", color = "white") +
  geom_vline(xintercept = pop_mean, color = "red", linewidth = 1) +
  scale_x_continuous(labels = scales::label_dollar()) +
  labs(
    title = "...",
    x = "...",
    y = "...",
    caption = "Source: Transparent California, via Data 8"
  ) +
  theme_minimal(base_size = 16)

salary_hist

In [ ]:
. <- ottr::check("tests/q1_1.R")

This is not a bell curve. The tall bar at the far left counts about 3,100 employees who earned less
than \$5,000 in 2023, and about 1,100 of them are listed at \$0. On the right, a few top salaries
stretch the axis past \$600,000. We keep every row: this file is the population we're studying, zeros
included.

### Drawing samples

`slice_sample(n = 5)` draws 5 random rows from a tibble. As in the
[Why the Mean Matters](https://dats.columbian.gwu.edu/ds4all/modules/mean.html) notes, we add
`replace = TRUE`, so every draw comes from the full population. Drawing a few hundred salaries out of
41,252 without replacement would give nearly the same results, because each draw hardly changes the
population.

Here is a random sample of 5 salaries:

In [ ]:
salaries |>
  slice_sample(n = 5, replace = TRUE)

Run the cell a few times. You get five different salaries each time.

---

**Question 1.2.** Draw one random sample of 100 salaries, with replacement, and compute its mean.
Store the mean as a single number in `one_mean`.

*Hint:* `summarize()` gives you a table with one row. `pull()` takes the number out of it.

Run your cell a few times. Each run draws a new sample of 100 employees, so each mean is a different
estimate of `pop_mean`.

In [ ]:
one_mean <- NULL # YOUR CODE HERE

one_mean

In [ ]:
. <- ottr::check("tests/q1_2.R")

How far from `pop_mean` can these estimates land? To find out, we draw thousands of samples. The
infer function `rep_slice_sample()` draws `reps` random samples of size `n` and stacks them into one
big table, with a column `replicate` that says which sample each row came from. The result is grouped
by `replicate`, so `summarize()` then computes one value per sample. Here are the means of 3 samples:

In [ ]:
salaries |>
  rep_slice_sample(n = 100, replace = TRUE, reps = 3) |>
  summarize(sample_mean = mean(salary))

---

**Question 1.3.** Use `rep_slice_sample()` and `summarize()` to simulate 10,000 sample means, each from
a sample of 100 salaries drawn with replacement. Store the result in `sample_means_100`: a tibble with
one row per sample, and the means in a column named `sample_mean`.

In [ ]:
sample_means_100 <- NULL # YOUR CODE HERE

sample_means_100

Run the code below to see the distribution of your 10,000 sample means. The red line is `pop_mean`
again.

In [ ]:
sample_means_100 |>
  ggplot(aes(x = sample_mean)) +
  geom_histogram(binwidth = 1000, fill = "chartreuse4", color = "white") +
  geom_vline(xintercept = pop_mean, color = "red", linewidth = 1) +
  scale_x_continuous(labels = scales::label_dollar()) +
  labs(
    title = "10,000 sample means, each from 100 salaries",
    x = "Sample mean salary",
    y = "Count",
    caption = "Source: Transparent California, via Data 8"
  ) +
  theme_minimal(base_size = 16)

The salaries themselves are far from a bell curve. Why is the histogram of sample means bell-shaped
anyway?

- `A.` The population of salaries is close to bell-shaped once you set aside the zeros.
- `B.` With 10,000 repetitions, a histogram of almost any statistic ends up looking like a bell curve.
- `C.` Each mean averages 100 salaries, so high and low ones balance out and the means pile up near the middle.
- `D.` Sampling with replacement gives every salary the same chance, which turns any population into a bell curve.

Set `bell_answer` to the letter, as a string.

In [ ]:
bell_answer <- NULL # YOUR CODE HERE

bell_answer

In [ ]:
. <- ottr::check("tests/q1_3.R")

This is the **Central Limit Theorem**: the sum or average of a large random sample drawn with
replacement has a roughly normal (bell-shaped) distribution, whatever the distribution of the
population it came from. The bell is centered at the population mean. The 1.96 in a 95% interval
comes from this normal curve, which is why the formula intervals from lecture work for skewed data like
these salaries, as long as the sample is large.

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


## 2. Bigger samples, less spread

The histogram in Question 1.3 shows how far a sample mean can land from `pop_mean` when the sample has
100 salaries. What happens with bigger samples?

---

**Question 2.1.** Simulate 10,000 sample means from samples of 400 salaries, and 10,000 more from
samples of 625, the same way as in Question 1.3. Store them in `sample_means_400` and
`sample_means_625`, each with the means in a column named `sample_mean`.

In [ ]:
sample_means_400 <- NULL # YOUR CODE HERE
sample_means_625 <- NULL # YOUR CODE HERE

c(nrow(sample_means_400), nrow(sample_means_625))

The plot below puts the three histograms on the same x-axis. `bind_rows()` stacks the three tibbles
into one, and `.id` adds a column saying which sample size each mean came from. `facet_wrap()` gives
each sample size its own panel.

In [ ]:
options(repr.plot.height = 6.5)  # a taller plot, to fit three panels

all_means <- bind_rows(
  `100 salaries` = sample_means_100,
  `400 salaries` = sample_means_400,
  `625 salaries` = sample_means_625,
  .id = "sample_size"
)

ggplot(all_means) +
  aes(x = sample_mean) +
  geom_histogram(binwidth = 1000, fill = "chartreuse4", color = "white") +
  geom_vline(xintercept = pop_mean, color = "red", linewidth = 1) +
  facet_wrap(~ sample_size, ncol = 1) +
  scale_x_continuous(labels = scales::label_dollar()) +
  labs(
    title = "Bigger samples, less spread",
    x = "Sample mean salary",
    y = "Count",
    caption = "Source: Transparent California, via Data 8"
  ) +
  theme_minimal(base_size = 16)

All three are centered at `pop_mean`, and the bigger the sample, the narrower the histogram. How much
narrower? Statistics gives a prediction:

$$
\text{SD of the sample means} = \frac{\text{population SD}}{\sqrt{\text{sample size}}}.
$$

Finish the table `sd_comparison` to compare the prediction with your simulations. Fill in the SDs of
your other two sets of sample means, and the prediction for each sample size.

In [ ]:
sd_comparison <- tibble(
  sample_size  = c(100, 400, 625),
  simulated_sd = c(
    sd(sample_means_100$sample_mean),
    ...,
    ...
  ),
  predicted_sd = ... / sqrt(sample_size)
)

sd_comparison

In [ ]:
. <- ottr::check("tests/q2_1.R")

The simulations and the formula agree. Going from 100 salaries to 400, four times the data, cuts the
SD in half, because $\sqrt{4} = 2$.

The SD of the sample means has its own name: the **standard error** (SE) of the sample mean. It
measures how far a sample mean typically lands from the population mean.

### Repetitions or sample size?

We ran 10,000 repetitions every time. Does that number matter? The first 1,000 rows of
`sample_means_100` are a simulation with 1,000 repetitions. Compare their SD with the SD of all
10,000:

In [ ]:
tibble(
  repetitions        = c(1000, 10000),
  sd_of_sample_means = c(
    sd(sample_means_100$sample_mean[1:1000]),
    sd(sample_means_100$sample_mean)
  )
)

---

**Question 2.2.** Which change would make the histogram of sample means **narrower**?

- `A.` Keep samples of 100 salaries, and run 50,000 repetitions instead of 10,000.
- `B.` Keep 10,000 repetitions, and draw samples of 625 salaries instead of 100.
- `C.` Either change works, since both of them use more salaries in total.
- `D.` Neither change works, since the spread depends only on the population.

Set `spread_answer` to the letter, as a string.

In [ ]:
spread_answer <- NULL # YOUR CODE HERE

spread_answer

In [ ]:
. <- ottr::check("tests/q2_2.R")

More repetitions give a smoother histogram and a steadier estimate of its SD, but the SD itself stays
about the same. Only a bigger sample shrinks it. The same goes for `reps` everywhere in `infer`:
raising it in `rep_slice_sample()` or `generate()` gives you a smoother distribution, not a smaller
standard error.

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


## 3. One sample, two intervals

So far we've had something you never get in practice: the whole population, to draw as many samples as
we liked. In practice you get one sample. From here on, pretend `salaries` is locked away, and all you
have is this sample of 400 employees:

In [ ]:
set.seed(2026)
our_sample <- salaries |> slice_sample(n = 400)

sample_mean <- mean(our_sample$salary)
sample_mean

`set.seed()` gives everyone the same sample, so we can compare answers. `sample_mean` is our estimate
of `pop_mean`. How far off could it be? By about one standard error, but the formula from Part 2 needs
the population SD, and that's locked away too. There are two ways around this.

1. **Bootstrap.** Treat `our_sample` as if it were the population. Resample 400 salaries from it, with
   replacement, and take the mean, thousands of times. The SD of those bootstrap means estimates the
   SE. `infer` does the resampling for you.
2. **Formula.** Use the SD of `our_sample` in place of the population SD:
   $\text{SE} \approx \dfrac{\text{SD of the sample}}{\sqrt{\text{sample size}}}.$

Notice the two kinds of sampling. `our_sample` is drawn *without* replacement, the default for
`slice_sample()`, because a real survey doesn't pick the same employee twice. The bootstrap resamples
from `our_sample` *with* replacement. Without replacement, every resample of 400 would be `our_sample`
again in a different order.

---

**Question 3.1.** Estimate the SE both ways.

- `boot_dist`: the bootstrap distribution of the mean salary in `our_sample`, from 5,000 repetitions.
  Fill in the blanks in the `infer` pipeline.
- `boot_se`: the SD of the `stat` column of `boot_dist`, as a single number.
- `formula_se`: the SD of the salaries in `our_sample`, divided by $\sqrt{400}$.

In [ ]:
boot_dist <- our_sample |>
  specify(response = ...) |>
  generate(reps = ..., type = "...") |>
  calculate(stat = "...")

boot_se    <- NULL # YOUR CODE HERE
formula_se <- NULL # YOUR CODE HERE

c(boot_se, formula_se)

In [ ]:
. <- ottr::check("tests/q3_1.R")

Because this is a lab, we can peek at the population and compare both estimates with the true SE from
Part 2:

In [ ]:
tibble(
  estimate = c("bootstrap, from our_sample", "formula, from our_sample", "true SE, from the population"),
  se       = c(boot_se, formula_se, pop_sd / sqrt(400))
)

All three are close, and the first two needed only the sample.

In Coding Assignment 4 you'll write both estimates as functions, for a proportion instead of a mean:
`SE_p_boot()` is the SD of the `stat` column, and `SE_p_hat()` is the formula. For a column of 0s and
1s where a proportion $p$ are 1s, the SD is about $\sqrt{p(1-p)}$, so SD / $\sqrt{n}$ becomes the
$\sqrt{p(1-p)/n}$ in `SE_p_hat()`.

### From SE to interval

Each SE turns into a 95% confidence interval.

- **Percentile interval:** the middle 95% of the bootstrap means, from the 2.5th to the 97.5th
  percentile. `quantile(x, 0.025)` finds the 2.5th percentile of `x`. (`get_ci()` from lecture does
  the same thing.)
- **Formula interval:** estimate $\pm$ 1.96 $\times$ SE, because 95% of a normal distribution lies
  within 1.96 SDs of its center. In R, `qnorm(0.975)` gives 1.959964.

---

**Question 3.2.** Build both 95% intervals as one-row tibbles with the columns `lower_ci` and
`upper_ci`, in that order:

- `ci_percentile`, from `boot_dist`; and
- `ci_formula`, from `sample_mean` and `formula_se`.

Fill in the blanks.

In [ ]:
ci_percentile <- boot_dist |>
  summarize(
    lower_ci = quantile(stat, ...),
    upper_ci = quantile(stat, ...)
  )

ci_formula <- tibble(
  lower_ci = ...,
  upper_ci = ...
)

bind_rows(ci_percentile, ci_formula)

In [ ]:
. <- ottr::check("tests/q3_2.R")

The two intervals nearly match. That's the Central Limit Theorem again: the bootstrap means are close
to a bell curve, so cutting off 2.5% on each side lands about 1.96 SEs from the center.

Since this is a lab, we can also ask what you can never ask of a real sample: does the interval contain
the true mean?

In [ ]:
ci_formula$lower_ci <= pop_mean & pop_mean <= ci_formula$upper_ci

This one does. With a real sample you would never find out. What you can know is how often the method
works, and that's Part 4.

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


## 4. How often is a 95% interval wrong?

"95% confident" describes the method, not any one interval. If we drew many samples and built an
interval from each, about 95% of the intervals would contain `pop_mean`. We have the population, so we
can check.

We'll use the formula interval, because it's fast. Data 8 runs the same check with bootstrap intervals
in [Section 13.2](https://inferentialthinking.com/chapters/13/2/bootstrap), and it takes several
minutes.

`rep_slice_sample()` can build all the intervals at once. Draw many samples, as in Part 1, and have
`summarize()` compute an interval for each sample instead of a mean.

---

**Question 4.1.** Fill in the blanks in `many_intervals()`. It should draw `reps` samples of size `n`
from `salaries`, with replacement, and compute each sample's 95% formula interval in the columns
`lower_ci` and `upper_ci`:

$$
\text{sample mean} \pm 1.96 \times \frac{\text{SD of the sample}}{\sqrt{n}}.
$$

In [ ]:
many_intervals <- function(n, reps) {
  salaries |>
    rep_slice_sample(n = ..., replace = TRUE, reps = ...) |>
    summarize(
      lower_ci = ...,
      upper_ci = ...
    )
}

many_intervals(400, 3)

Now make 100 intervals, each from its own sample of 400 salaries. Add a column `covers` that is `TRUE`
when an interval contains `pop_mean` and `FALSE` when it misses. Then count the intervals that contain
`pop_mean`, and store the count in `num_covered`.

In [ ]:
intervals <- many_intervals(400, 100) |>
  mutate(covers = ...)

num_covered <- NULL # YOUR CODE HERE

num_covered

In [ ]:
. <- ottr::check("tests/q4_1.R")

Each horizontal line below is one interval. The black line is `pop_mean`, which never moves. The
intervals move around it from sample to sample, and the red ones miss it.

In [ ]:
intervals |>
  ggplot() +
  geom_segment(
    aes(x = lower_ci, xend = upper_ci, y = replicate, yend = replicate, color = covers),
    linewidth = 1
  ) +
  geom_vline(xintercept = pop_mean, linewidth = 1) +
  scale_color_manual(
    values = c(`TRUE` = "gray60", `FALSE` = "red"),
    breaks = c("TRUE", "FALSE"),
    labels = c("contains pop_mean", "misses pop_mean")
  ) +
  scale_x_continuous(labels = scales::label_dollar()) +
  labs(
    title = "100 intervals, each from a sample of 400",
    x = "Average salary",
    y = "Sample",
    color = NULL,
    caption = "Source: Transparent California, via Data 8"
  ) +
  theme_minimal(base_size = 16) +
  theme(legend.position = "bottom", plot.margin = margin(5.5, 25, 5.5, 5.5))

You should see about 5 red intervals out of 100. Run the cell that makes `intervals` again and the
count changes, but it stays near 95 out of 100.

### Using an interval to test a claim

A 95% interval can test a claim about the population: if the claimed value falls outside the interval,
reject the claim. This makes the same decision as a test with a p-value cutoff of 0.05, when the
alternative is that the true value differs from the claim in either direction. You'll use it in Task 10
of Coding Assignment 4.

---

**Question 4.2.** Suppose someone claims the average salary of all 41,252 employees equals `pop_mean`,
so the claim is true. You draw one sample of 400, build its 95% interval the way `many_intervals()`
does, and reject the claim if the interval does not contain `pop_mean`. What is the chance that you
reject this true claim?

- `A.` 0%, because a true claim always lands inside the interval.
- `B.` About 95%, because that is the confidence level we chose.
- `C.` About 5% with samples of 100, but close to 0% with samples of 625.
- `D.` About 5%, because that is how often a 95% interval misses.

Set `error_answer` to the letter, as a string.

In [ ]:
error_answer <- NULL # YOUR CODE HERE

error_answer

In [ ]:
. <- ottr::check("tests/q4_2.R")

This is an **error probability**: the chance of rejecting a claim that is true. With a 95% interval,
it is 5%, the same as a p-value cutoff of 0.05, and a bigger sample doesn't change it. To make this
error rarer, you would use a higher confidence level, like 99%, and every interval would get wider.

### The other kind of error

The other mistake is failing to reject a claim that is false. Suppose someone claims the average salary
was \$125,000. We know that's wrong: `pop_mean` is well below it. How often does the interval method
catch the mistake?

The cell below builds 1,000 intervals from samples of 100, and 1,000 more from samples of 625.

In [ ]:
claim <- 125000

intervals_100 <- many_intervals(100, 1000)
intervals_625 <- many_intervals(625, 1000)

---

**Question 4.3.** An interval rejects the claim when `claim` is below `lower_ci` or above `upper_ci`.
For each sample size, compute the share of the 1,000 intervals that reject the claim, as a proportion
between 0 and 1. Store them in `reject_rate_100` and `reject_rate_625`.

*Hint:* `|` means "or". As in Lab 4, the mean of a column of `TRUE`/`FALSE` values is the share that
are `TRUE`.

In [ ]:
reject_rate_100 <- NULL # YOUR CODE HERE
reject_rate_625 <- NULL # YOUR CODE HERE

c(reject_rate_100, reject_rate_625)

Why do samples of 625 reject the false claim more often?

- `A.` Their intervals are narrower, so fewer of them stretch far enough to reach \$125,000.
- `B.` Their intervals are wider, so more of them end up far away from \$125,000.
- `C.` They use a lower confidence level, so their intervals miss more often.
- `D.` Their sample means tend to land farther from the population mean.

Set `why_answer` to the letter, as a string.

In [ ]:
why_answer <- NULL # YOUR CODE HERE

why_answer

In [ ]:
. <- ottr::check("tests/q4_3.R")

With samples of 100, the false claim usually slips through. With samples of 625, it rarely does. A
bigger sample can't lower the 5% chance of rejecting a true claim, but it does make you more likely to
reject a false one. Statisticians call rejecting a true claim a *Type I error*, and failing to reject a
false claim a *Type II error*.

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


You're done with Lab 5!

Here's what you can now do: simulate the distribution of a sample mean, explain why it's bell-shaped
and how its spread depends on the sample size, estimate a standard error and build a 95% interval from
one sample two ways, and say what "95%" promises: a method that is wrong about 5% of the time.

**Important submission information:** Be sure to run the tests and verify that they all pass, then
choose **Save Notebook** from the **File** menu, then Download the .ipynb. Then, go to your course's
Gradescope and submit the corresponding assignment. The name of this assignment is "Lab 05".


### Moral support of this data science course

Good work! Just a cat guarding his pumpkin kingdom

<img src="pumpkins.jpg" alt="cat sitting next to pumpkins" width="300"/>

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
# Re-run all the autograder tests in the tests/ folder.
. <- sapply(list.files("tests", pattern = "\\.R$", full.names = TRUE), function(f) ottr::check(f))

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all
images/graphs appear in the output. Click Download in the bar at the top of the notebook. **Please save
before downloading!** Submit your final version to Gradescope.